# EOB Text Extractor — Colab Demo

# ⚠️ SYNTHETIC DATA ONLY. DO NOT UPLOAD REAL EOBs.

**Google Colab is NOT HIPAA-compliant.** This notebook runs entirely on
obviously-fake synthetic EOB data generated on the fly. **Never upload,
paste, or process real Explanation of Benefits documents or any real
Protected Health Information (PHI) here.**

For real EOBs, run this tool on a HIPAA-compliant local environment with
appropriate safeguards (encryption, access controls, audit logging).

In [ ]:
# Cell 1 — fetch the project so requirements.txt and the scripts are present.
# Colab starts with an empty working directory, so the repo must be cloned
# before any of the cells below can find requirements.txt.
import os

REPO_URL = 'https://github.com/ThirdPartyThinker/EOB-Text-Extractor.git'
REPO_DIR = 'EOB-Text-Extractor'

if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL
%cd $REPO_DIR
!ls

In [ ]:
# Cell 2 — install system dependencies (Tesseract OCR engine + poppler)
!apt-get update && apt-get install -y tesseract-ocr poppler-utils

In [ ]:
# Cell 3 — install pinned Python dependencies
!pip install -r requirements.txt

In [ ]:
# Cell 4 — generate synthetic (fake) EOB PDFs into ./synthetic_in/
!python generate_synthetic_eobs.py --output ./synthetic_in
!ls -l ./synthetic_in

In [ ]:
# Cell 5 — batch-process the synthetic PDFs into ./synthetic_out/
!python run_batch.py --input ./synthetic_in --output ./synthetic_out
!ls -l ./synthetic_out

In [ ]:
# Cell 6 — display a few example outputs and a sidecar JSON
import json
from pathlib import Path

out = Path('./synthetic_out')
for txt in sorted(out.glob('*.txt'))[:2]:
    print('=' * 70)
    print('TEXT OUTPUT:', txt.name)
    print('=' * 70)
    print(txt.read_text())

    sidecar = txt.with_suffix('.json')
    print('-' * 70)
    print('SIDECAR METADATA (no PHI):', sidecar.name)
    print(json.dumps(json.loads(sidecar.read_text()), indent=2))
    print()

# Optional: best-effort structured field extraction
from eob_extract import extract_eob_fields
sample = sorted(out.glob('*.txt'))[0].read_text()
print('=' * 70)
print('BEST-EFFORT FIELD EXTRACTION (varies by payer template):')
print(json.dumps(extract_eob_fields(sample), indent=2))